# 4-Configuration Evaluation: Legal Demand Assistant

Compares all four adaptation configurations against the 10 held-out evaluation queries:

| Config | Description |
|---|---|
| **Baseline** | Base Qwen3.5-9B, generic prompt |
| **GEPA** | Base model + GEPA-optimized prompts |
| **QLoRA** | Fine-tuned model + generic prompt |
| **GEPA + QLoRA** | Fine-tuned model + GEPA-optimized prompts |

**Evaluation criteria:** Accuracy, Domain Relevance, Hallucination Rate, Clarity, Structure

**Requirements:** Colab with GPU (A100 recommended). Upload the QLoRA adapter zip and GEPA prompts JSON.

## Section 0: Install Dependencies

In [ ]:
!pip install transformers>=4.46.0 peft>=0.13.0 bitsandbytes>=0.44.0 accelerate

## Section 1: Imports and Configuration

In [ ]:
import json
import os
from collections import defaultdict

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3.5-9B"
ADAPTER_DIR = "./qlora-legal-demand-adapter"  # Update after uploading/unzipping
GEPA_PROMPTS_FILE = "./gepa_optimized_prompts.json"  # Update after uploading

BASELINE_PROMPT = (
    "You are a legal demand assistant. You help draft demand letters, "
    "identify legal claims, extract elements from demand letters, "
    "evaluate letter quality, and recommend remedies."
)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## Section 2: Upload Artifacts

Upload the QLoRA adapter and GEPA prompts.

In [ ]:
# Option A: Upload via Colab UI
# from google.colab import files
# uploaded = files.upload()  # Upload qlora-legal-demand-adapter.zip and gepa_optimized_prompts.json

# Option B: Clone repo (uncomment and update URL)
# !git clone https://github.com/BigDataAnalytics-CS5542/Lab_8.git

# Unzip adapter if needed
# !unzip -o qlora-legal-demand-adapter.zip

print("Upload qlora-legal-demand-adapter.zip and gepa_optimized_prompts.json, then run the cells above.")

## Section 3: Load GEPA Prompts

In [ ]:
# Load GEPA-optimized prompts
if os.path.exists(GEPA_PROMPTS_FILE):
    with open(GEPA_PROMPTS_FILE) as f:
        gepa_data = json.load(f)
    gepa_prompts = gepa_data["optimized_prompts"]
    print(f"GEPA prompts loaded from {GEPA_PROMPTS_FILE}")
    print(f"GEPA optimization score: {gepa_data['metadata'].get('final_score', 'N/A')}")
    for task, prompt in gepa_prompts.items():
        print(f"  {task}: {prompt[:80]}...")
else:
    print(f"WARNING: {GEPA_PROMPTS_FILE} not found.")
    print("Using placeholder prompts — replace with actual GEPA results.")
    # Placeholder prompts — REPLACE THESE with your GEPA output
    gepa_prompts = {
        "draft_demand_letter": "PLACEHOLDER — paste your GEPA-optimized prompt here",
        "identify_claim": "PLACEHOLDER — paste your GEPA-optimized prompt here",
        "extract_elements": "PLACEHOLDER — paste your GEPA-optimized prompt here",
        "evaluate_letter": "PLACEHOLDER — paste your GEPA-optimized prompt here",
        "recommend_remedy": "PLACEHOLDER — paste your GEPA-optimized prompt here",
    }

## Section 4: Load Models

Load the base model and the QLoRA-adapted model.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load base model
base_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    attn_implementation="eager",
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
base_model.eval()
print(f"Base model loaded: {MODEL_ID}")
print(f"Memory: {base_model.get_memory_footprint() / 1e9:.2f} GB")

In [ ]:
# Load QLoRA model (base + adapter)
if os.path.exists(ADAPTER_DIR):
    qlora_tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
    if qlora_tokenizer.pad_token is None:
        qlora_tokenizer.pad_token = qlora_tokenizer.eos_token

    qlora_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
    qlora_model.eval()
    print(f"QLoRA adapter loaded from {ADAPTER_DIR}")
else:
    print(f"WARNING: Adapter not found at {ADAPTER_DIR}")
    print("QLoRA configs will use base model as fallback.")
    qlora_model = base_model
    qlora_tokenizer = base_tokenizer

## Section 5: Inference and Scoring Functions

In [ ]:
def generate(model, tokenizer, system_prompt, user_input, max_tokens=512):
    """Generate a response from a model."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_tokens, temperature=0.7,
            top_p=0.9, do_sample=True,
        )
    return tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()


def score_response(response, criteria):
    """Score a response against criteria. Returns per-criterion dict and total."""
    scores = {}
    resp_lower = response.lower()

    for c in criteria:
        if c == "formal_tone":
            scores[c] = any(w in resp_lower for w in ["dear", "sincerely", "represent", "i represent"])
        elif c == "parties_identified":
            scores[c] = "[" in response or any(w in resp_lower for w in ["my client", "claimant", "tenant", "freelancer"])
        elif c == "amount_stated":
            scores[c] = "$" in response
        elif c == "deadline_included":
            scores[c] = any(w in resp_lower for w in ["days", "deadline", "within"])
        elif c == "escalation_mentioned":
            scores[c] = any(w in resp_lower for w in ["legal action", "legal remedies", "pursue", "file", "court"])
        elif c == "legal_basis":
            scores[c] = any(w in resp_lower for w in ["breach", "violation", "obligation", "liability", "negligence", "statute"])
        elif c == "correct_claim_type":
            scores[c] = any(w in resp_lower for w in ["breach", "fraud", "misrepresentation", "negligence", "violation"])
        elif c == "json_format":
            scores[c] = "{" in response and "}" in response
        elif c == "reasoning_provided":
            scores[c] = any(w in resp_lower for w in ["reasoning", "because", "the ", "failed", "accepted"])
        elif c == "all_fields_extracted":
            scores[c] = all(w in resp_lower for w in ["claimant", "recipient", "damages", "deadline"])
        elif c == "values_correct":
            scores[c] = "$" in response
        elif c == "correctly_identifies_inadequate":
            scores[c] = any(w in resp_lower for w in ["false", "inadequate", "not adequate", "incomplete", "missing"])
        elif c == "correctly_identifies_adequate":
            scores[c] = any(w in resp_lower for w in ["true", "adequate", "effective", "complete", "sufficient"])
        elif c == "missing_elements_listed":
            scores[c] = any(w in resp_lower for w in ["missing", "lacks", "does not include", "no "])
        elif c == "improvements_suggested":
            scores[c] = any(w in resp_lower for w in ["should", "add", "include", "specify", "improve"])
        elif c == "strengths_listed":
            scores[c] = any(w in resp_lower for w in ["strength", "identifies", "includes", "states", "provides"])
        elif c == "multiple_remedies":
            scores[c] = resp_lower.count("remedy") + resp_lower.count("refund") + resp_lower.count("payment") + resp_lower.count("reimbursement") >= 2
        else:
            scores[c] = True

    total = sum(scores.values()) / len(scores) if scores else 0
    return scores, total

## Section 6: Evaluation Queries

In [ ]:
EVAL_QUERIES = [
    {
        "id": "Q1", "task_type": "draft_demand_letter",
        "input": "Draft a demand letter for a tenant whose landlord withheld a $2,500 security deposit without providing an itemized statement of damages.",
        "criteria": ["formal_tone", "parties_identified", "amount_stated", "deadline_included", "escalation_mentioned", "legal_basis"]
    },
    {
        "id": "Q2", "task_type": "draft_demand_letter",
        "input": "Draft a demand letter for a freelancer who completed branding work for a client, delivered the files, and never received the agreed $3,200 payment.",
        "criteria": ["formal_tone", "parties_identified", "amount_stated", "deadline_included", "escalation_mentioned", "legal_basis"]
    },
    {
        "id": "Q3", "task_type": "identify_claim",
        "input": "Identify the strongest legal claim where a contractor accepted an $8,000 payment, performed minimal work, and abandoned the job.",
        "criteria": ["correct_claim_type", "json_format", "reasoning_provided"]
    },
    {
        "id": "Q4", "task_type": "identify_claim",
        "input": "Identify the likely legal claim where a seller knowingly lied about a car having no accident history, and the buyer later discovered major prior damage.",
        "criteria": ["correct_claim_type", "json_format", "reasoning_provided"]
    },
    {
        "id": "Q5", "task_type": "extract_elements",
        "input": "Extract the claimant, recipient, damages, deadline, and requested remedy from the following demand letter:\n\nDear Mr. Allen: My client, Sarah Kim, seeks payment of $4,800 for water damage caused to her property on January 8, 2026, when your negligence led to a plumbing overflow from your unit. Please remit payment within 14 days of this letter to avoid further action.",
        "criteria": ["all_fields_extracted", "json_format", "values_correct"]
    },
    {
        "id": "Q6", "task_type": "extract_elements",
        "input": "Extract the key legal elements from the following demand letter:\n\nDear ABC Services, I represent Daniel Ortiz regarding unpaid wages in the amount of $1,950 for work completed during December 2025. Unless payment is received within 7 days, my client will consider further legal action.",
        "criteria": ["all_fields_extracted", "json_format", "values_correct"]
    },
    {
        "id": "Q7", "task_type": "evaluate_letter",
        "input": "Evaluate whether this demand letter is complete and effective:\n\nYou owe me money. Please fix this immediately.",
        "criteria": ["correctly_identifies_inadequate", "json_format", "missing_elements_listed", "improvements_suggested"]
    },
    {
        "id": "Q8", "task_type": "evaluate_letter",
        "input": "Evaluate whether this demand letter is effective:\n\nDear Property Manager, my client seeks return of a $1,800 security deposit for Apartment 2B. The tenant vacated the unit on February 1, 2026, and no itemized deductions were provided. Please return the deposit within 10 days or my client may pursue legal remedies.",
        "criteria": ["correctly_identifies_adequate", "json_format", "strengths_listed"]
    },
    {
        "id": "Q9", "task_type": "recommend_remedy",
        "input": "Suggest appropriate remedies when a contractor took a deposit, abandoned a home repair project, and the homeowner had to hire another contractor for additional cost.",
        "criteria": ["multiple_remedies", "json_format", "reasoning_provided"]
    },
    {
        "id": "Q10", "task_type": "recommend_remedy",
        "input": "Suggest remedies for a consumer who purchased a defective appliance, repeatedly requested repair or refund, and received no response from the seller.",
        "criteria": ["multiple_remedies", "json_format", "reasoning_provided"]
    },
]

## Section 7: Define the 4 Configurations

In [ ]:
def get_prompt_for_query(query, prompt_source):
    """Get the appropriate system prompt for a query."""
    if prompt_source == "baseline":
        return BASELINE_PROMPT
    elif prompt_source == "gepa":
        return gepa_prompts.get(query["task_type"], BASELINE_PROMPT)
    else:
        return BASELINE_PROMPT


CONFIGS = {
    "baseline": {
        "label": "Baseline",
        "model": base_model,
        "tokenizer": base_tokenizer,
        "prompt_source": "baseline",
    },
    "gepa": {
        "label": "GEPA",
        "model": base_model,
        "tokenizer": base_tokenizer,
        "prompt_source": "gepa",
    },
    "qlora": {
        "label": "QLoRA",
        "model": qlora_model,
        "tokenizer": qlora_tokenizer,
        "prompt_source": "baseline",
    },
    "gepa_qlora": {
        "label": "GEPA + QLoRA",
        "model": qlora_model,
        "tokenizer": qlora_tokenizer,
        "prompt_source": "gepa",
    },
}

print("Configurations:")
for key, cfg in CONFIGS.items():
    is_qlora = cfg["model"] is not base_model
    print(f"  {cfg['label']:15s} | model={'QLoRA' if is_qlora else 'Base':6s} | prompt={cfg['prompt_source']}")

## Section 8: Run Evaluation

Run all 10 queries against all 4 configurations (40 total inferences).

In [ ]:
all_results = {}  # config_key → [{query_id, response, scores, total}, ...]

for config_key, cfg in CONFIGS.items():
    print(f"\n{'=' * 60}")
    print(f"Running: {cfg['label']}")
    print(f"{'=' * 60}")

    config_results = []
    for q in EVAL_QUERIES:
        system_prompt = get_prompt_for_query(q, cfg["prompt_source"])
        response = generate(cfg["model"], cfg["tokenizer"], system_prompt, q["input"])
        scores, total = score_response(response, q["criteria"])

        config_results.append({
            "query_id": q["id"],
            "task_type": q["task_type"],
            "response": response,
            "criteria_scores": scores,
            "total_score": total,
        })
        print(f"  {q['id']} ({q['task_type'][:15]:15s}): {total:.0%} — {scores}")

    all_results[config_key] = config_results
    avg = sum(r["total_score"] for r in config_results) / len(config_results)
    print(f"  Average: {avg:.0%}")

print("\nAll evaluations complete.")

## Section 9: Comparison Table

In [ ]:
# Build comparison table
print(f"\n{'Query':<8}", end="")
for cfg in CONFIGS.values():
    print(f"{cfg['label']:>15s}", end="")
print()
print("-" * 68)

config_totals = defaultdict(list)

for i, q in enumerate(EVAL_QUERIES):
    print(f"{q['id']:<8}", end="")
    for config_key, cfg in CONFIGS.items():
        score = all_results[config_key][i]["total_score"]
        config_totals[config_key].append(score)
        print(f"{score:>14.0%} ", end="")
    print()

print("-" * 68)
print(f"{'Average':<8}", end="")
for config_key in CONFIGS:
    avg = sum(config_totals[config_key]) / len(config_totals[config_key])
    print(f"{avg:>14.0%} ", end="")
print()

## Section 10: Per-Task-Type Breakdown

In [ ]:
task_types = ["draft_demand_letter", "identify_claim", "extract_elements", "evaluate_letter", "recommend_remedy"]

print(f"\n{'Task Type':<25}", end="")
for cfg in CONFIGS.values():
    print(f"{cfg['label']:>15s}", end="")
print()
print("-" * 85)

for task_type in task_types:
    print(f"{task_type:<25}", end="")
    for config_key in CONFIGS:
        task_scores = [
            r["total_score"] for r in all_results[config_key]
            if r["task_type"] == task_type
        ]
        avg = sum(task_scores) / len(task_scores) if task_scores else 0
        print(f"{avg:>14.0%} ", end="")
    print()

## Section 11: Side-by-Side Response Comparison

View actual responses for each query across all 4 configs.

In [ ]:
for i, q in enumerate(EVAL_QUERIES):
    print(f"\n{'#' * 70}")
    print(f"  {q['id']} — {q['task_type']}")
    print(f"  Input: {q['input'][:100]}...")
    print(f"{'#' * 70}")

    for config_key, cfg in CONFIGS.items():
        result = all_results[config_key][i]
        print(f"\n  [{cfg['label']}] Score: {result['total_score']:.0%}")
        print(f"  {'-' * 50}")
        # Print response, truncated for readability
        resp = result["response"]
        if len(resp) > 400:
            resp = resp[:400] + "..."
        for line in resp.split("\n"):
            print(f"  {line}")

## Section 12: Save Results

In [ ]:
# Compute summary metrics
summary = {}
for config_key, cfg in CONFIGS.items():
    results = all_results[config_key]
    avg_total = sum(r["total_score"] for r in results) / len(results)

    # Per task type
    per_task = {}
    for task_type in task_types:
        task_results = [r for r in results if r["task_type"] == task_type]
        if task_results:
            per_task[task_type] = sum(r["total_score"] for r in task_results) / len(task_results)

    summary[config_key] = {
        "label": cfg["label"],
        "overall_score": avg_total,
        "per_task": per_task,
        "per_query": {r["query_id"]: r["total_score"] for r in results},
    }

output = {
    "metadata": {
        "model": MODEL_ID,
        "timestamp": str(datetime.now()) if 'datetime' not in dir() else __import__('datetime').datetime.now().isoformat(),
        "num_queries": len(EVAL_QUERIES),
        "configs": list(CONFIGS.keys()),
    },
    "summary": summary,
    "detailed_results": {
        config_key: [
            {
                "query_id": r["query_id"],
                "task_type": r["task_type"],
                "response": r["response"],
                "criteria_scores": r["criteria_scores"],
                "total_score": r["total_score"],
            }
            for r in results
        ]
        for config_key, results in all_results.items()
    },
}

with open("eval_results.json", "w") as f:
    json.dump(output, f, indent=2)

print("Results saved to eval_results.json")
print("\nFinal Summary:")
for config_key, data in summary.items():
    print(f"  {data['label']:15s}: {data['overall_score']:.0%}")

## Section 13: Generate Results Markdown

Auto-generate the `results.md` table for the lab report.

In [ ]:
md = "# Evaluation Results\n\n"
md += "## Evaluation Queries\n"
md += "10 queries from evaluation_queries.md were used to compare system configurations.\n\n"

md += "## Comparison Table\n\n"
md += "| Query | Baseline | GEPA | QLoRA | GEPA + QLoRA |\n"
md += "|---|---|---|---|---|\n"
for i, q in enumerate(EVAL_QUERIES):
    scores = []
    for config_key in ["baseline", "gepa", "qlora", "gepa_qlora"]:
        s = all_results[config_key][i]["total_score"]
        scores.append(f"{s:.0%}")
    md += f"| {q['id']} | {' | '.join(scores)} |\n"

# Add averages row
avg_scores = []
for config_key in ["baseline", "gepa", "qlora", "gepa_qlora"]:
    avg = sum(r["total_score"] for r in all_results[config_key]) / len(all_results[config_key])
    avg_scores.append(f"**{avg:.0%}**")
md += f"| **Average** | {' | '.join(avg_scores)} |\n"

md += "\n## Per-Task Breakdown\n\n"
md += "| Task Type | Baseline | GEPA | QLoRA | GEPA + QLoRA |\n"
md += "|---|---|---|---|---|\n"
for task_type in task_types:
    row_scores = []
    for config_key in ["baseline", "gepa", "qlora", "gepa_qlora"]:
        task_results = [r for r in all_results[config_key] if r["task_type"] == task_type]
        avg = sum(r["total_score"] for r in task_results) / len(task_results) if task_results else 0
        row_scores.append(f"{avg:.0%}")
    md += f"| {task_type} | {' | '.join(row_scores)} |\n"

md += "\n## Observations\n\n"

# Auto-generate observations based on results
baseline_avg = sum(r["total_score"] for r in all_results["baseline"]) / len(all_results["baseline"])
gepa_avg = sum(r["total_score"] for r in all_results["gepa"]) / len(all_results["gepa"])
qlora_avg = sum(r["total_score"] for r in all_results["qlora"]) / len(all_results["qlora"])
combined_avg = sum(r["total_score"] for r in all_results["gepa_qlora"]) / len(all_results["gepa_qlora"])

best_config = max([("Baseline", baseline_avg), ("GEPA", gepa_avg), ("QLoRA", qlora_avg), ("GEPA + QLoRA", combined_avg)], key=lambda x: x[1])

md += f"- **Best configuration:** {best_config[0]} ({best_config[1]:.0%})\n"
md += f"- GEPA prompt optimization improved over baseline by {gepa_avg - baseline_avg:+.0%}\n"
md += f"- QLoRA fine-tuning improved over baseline by {qlora_avg - baseline_avg:+.0%}\n"
md += f"- Combined GEPA + QLoRA improved over baseline by {combined_avg - baseline_avg:+.0%}\n"

print(md)

with open("results.md", "w") as f:
    f.write(md)
print("\nSaved to results.md")

## Section 14: Download Results

In [ ]:
# Uncomment to download from Colab
# from google.colab import files
# files.download("eval_results.json")
# files.download("results.md")